# 4. ONNX export

Export the dense-RoPE encoder/cross-KV graph and one cached decoder-step graph. CPU algebra checks compare encoder features and all 25 decoder positions on synthetic inputs before writing deployment metadata. These checks are not a dataset accuracy or TensorRT qualification.

In [ ]:
from pathlib import Path
import os, sys, subprocess, json
ROOT = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "waterec_ocr").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook inside the repository")
sys.path.insert(0, str(ROOT))
from waterec_ocr.waterec_config import load_config
from waterec_ocr.waterec_models import default_checkpoint
CONFIG_PATH = os.environ.get("WATEREC_CONFIG", "configs/dummy_alpr.yaml")
config = load_config(CONFIG_PATH)
checkpoint = default_checkpoint(config)
deployment = config["output_dir"] / "deployment"
examples = sorted((config["dataset_root"] / "val").glob("*.png"))
IMAGE_PATH = examples[0] if examples else config["dataset_root"] / "val/example.png"
def run(*args):
    return subprocess.run([sys.executable, "-m", "waterec_ocr.waterec_cli", *args, "--config", CONFIG_PATH], cwd=ROOT, check=True)
print("Profile:", CONFIG_PATH, "variant:", config["variant"], "patch budget:", config["tokens"])


In [ ]:
RUN_EXPORT = os.environ.get("RUN_EXPORT", "0") == "1"
if RUN_EXPORT:
    run("export")
else:
    print("Enable RUN_EXPORT after training. Destination:", deployment)


Keep the checkpoint, configuration, vocabulary, preprocessing policy, RoPE table and exported graphs together. Each resolution budget needs its matching export. The standard deployment is static batch 1. Normalization/attention safeguards and explicit intermediate outputs preserve the tested runtime policy; export success alone does not prove FP16 prediction parity.